In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("okienka").getOrCreate()
spark

In [2]:
czujnik_temperatury = ((12.5, "2019-01-02 12:00:00"),
(17.6, "2019-01-02 12:00:20"),
(14.6,  "2019-01-02 12:00:30"),
(22.9,  "2019-01-02 12:01:15"),
(17.4,  "2019-01-02 12:01:30"),
(25.8,  "2019-01-02 12:03:25"),
(27.1,  "2019-01-02 12:02:40"),
)

In [3]:
from pyspark.sql.functions import to_timestamp
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

schema = StructType([
    StructField("temperatura", DoubleType(), True),
    StructField("czas", StringType(), True),
])

In [4]:
df = (spark.createDataFrame(czujnik_temperatury, schema=schema)
      .withColumn("czas", to_timestamp("czas")))

df.printSchema()

root
 |-- temperatura: double (nullable = true)
 |-- czas: timestamp (nullable = true)



In [5]:
df.show(3)

+-----------+-------------------+
|temperatura|               czas|
+-----------+-------------------+
|       12.5|2019-01-02 12:00:00|
|       17.6|2019-01-02 12:00:20|
|       14.6|2019-01-02 12:00:30|
+-----------+-------------------+
only showing top 3 rows



In [6]:
df.createOrReplaceTempView("df")

In [7]:
spark.sql("select czas, temperatura from df where temperatura > 21").show(5)

+-------------------+-----------+
|               czas|temperatura|
+-------------------+-----------+
|2019-01-02 12:01:15|       22.9|
|2019-01-02 12:03:25|       25.8|
|2019-01-02 12:02:40|       27.1|
+-------------------+-----------+



In [8]:
# Thumbling window

import pyspark.sql.functions as F

df2 = df.groupBy(F.window("czas","30 seconds")).count()
df2.show(truncate=False)

+------------------------------------------+-----+
|window                                    |count|
+------------------------------------------+-----+
|{2019-01-02 12:00:00, 2019-01-02 12:00:30}|2    |
|{2019-01-02 12:00:30, 2019-01-02 12:01:00}|1    |
|{2019-01-02 12:01:00, 2019-01-02 12:01:30}|1    |
|{2019-01-02 12:01:30, 2019-01-02 12:02:00}|1    |
|{2019-01-02 12:03:00, 2019-01-02 12:03:30}|1    |
|{2019-01-02 12:02:30, 2019-01-02 12:03:00}|1    |
+------------------------------------------+-----+



In [9]:
df2.printSchema()

root
 |-- window: struct (nullable = false)
 |    |-- start: timestamp (nullable = true)
 |    |-- end: timestamp (nullable = true)
 |-- count: long (nullable = false)



In [10]:
spark.stop()

In [ ]:
####### cz2

In [11]:
df = spark.readStream.format("rate").option("rowsPerSecond", 1).load()

Py4JJavaError: An error occurred while calling o58.load.
: org.apache.spark.SparkException: [INTERNAL_ERROR] No active or default Spark session found SQLSTATE: XX000
	at org.apache.spark.SparkException$.internalError(SparkException.scala:92)
	at org.apache.spark.SparkException$.internalError(SparkException.scala:96)
	at org.apache.spark.sql.SparkSession$.$anonfun$active$2(SparkSession.scala:1064)
	at scala.Option.getOrElse(Option.scala:201)
	at org.apache.spark.sql.SparkSession$.$anonfun$active$1(SparkSession.scala:1064)
	at scala.Option.getOrElse(Option.scala:201)
	at org.apache.spark.sql.SparkSession$.active(SparkSession.scala:1063)
	at org.apache.spark.sql.execution.streaming.sources.RateStreamProvider.getTable(RateStreamProvider.scala:66)
	at org.apache.spark.sql.internal.connector.SimpleTableProvider.getOrLoadTable(SimpleTableProvider.scala:35)
	at org.apache.spark.sql.internal.connector.SimpleTableProvider.inferSchema(SimpleTableProvider.scala:41)
	at org.apache.spark.sql.interna

In [12]:
%%file streamrate.py
## uruchom przez spark-submit streamrate.py

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

df = (spark.readStream
      .format("rate")
      .option("rowsPerSecond", 1)
      .load()
)


query = (df.writeStream 
    .format("console") 
    .outputMode("append") 
    .option("truncate", False) 
    .start()
) 

query.awaitTermination()

Writing streamrate.py


In [13]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

def process_batch(df, batch_id, tstop=5):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()


df = (spark.readStream
      .format("rate")
      .option("rowsPerSecond", 1)
      .load()
)

query = (df.writeStream 
    .format("console") 
    .outputMode("append")
    .foreachBatch(process_batch)
    .option("truncate", False) 
    .start()
)

Batch ID: 0
+---------+-----+
|timestamp|value|
+---------+-----+
+---------+-----+

Batch ID: 1
+----------------------+-----+
|timestamp             |value|
+----------------------+-----+
|2026-04-21 11:41:10.82|0    |
+----------------------+-----+

Batch ID: 2
+----------------------+-----+
|timestamp             |value|
+----------------------+-----+
|2026-04-21 11:41:11.82|1    |
+----------------------+-----+

Batch ID: 3
+----------------------+-----+
|timestamp             |value|
+----------------------+-----+
|2026-04-21 11:41:12.82|2    |
+----------------------+-----+

Batch ID: 4
+----------------------+-----+
|timestamp             |value|
+----------------------+-----+
|2026-04-21 11:41:13.82|3    |
+----------------------+-----+

Batch ID: 5
+----------------------+-----+
|timestamp             |value|
+----------------------+-----+
|2026-04-21 11:41:14.82|4    |
+----------------------+-----+

Batch ID: 0
+----+-----------+
|czas|temperatura|
+----+-----------+
+----+

In [ ]:
## symulator

In [14]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

def process_batch(df, batch_id, tstop=5):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()

from pyspark.sql.functions import col, expr

df = (spark.readStream
      .format("rate")
      .option("rowsPerSecond", 1)
      .load()
)

stream = (df.withColumn("czas", col("timestamp"))
        .withColumn("temperatura", expr("20 + rand() * 10"))
        .select("czas", "temperatura")
       )

query = (stream.writeStream 
    .format("console") 
    .outputMode("append")
    .foreachBatch(process_batch)
    .option("truncate", False) 
    .start()
)

In [ ]:
# 1. Transformacja bezstanowa (stateless transformation)

In [15]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

def process_batch(df, batch_id, tstop=5):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()

from pyspark.sql.functions import col, expr

df = (spark.readStream
      .format("rate")
      .option("rowsPerSecond", 1)
      .load()
)

stream = (df.withColumn("czas", col("timestamp"))
        .withColumn("temperatura", expr("20 + rand() * 10"))
        .select("czas", "temperatura")
       )

stream_filtered = stream.filter(col("temperatura") > 28)

query = (stream_filtered.writeStream 
    .format("console") 
    .outputMode("append")
    .foreachBatch(process_batch)
    .option("truncate", False) 
    .start()
)

In [ ]:
# 2. Prosty model wykrywający anomalie (bezstanowy)

In [16]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr, when

spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

def process_batch(df, batch_id, tstop=10):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()



df = (spark.readStream
      .format("rate")
      .option("rowsPerSecond", 1)
      .load()
)

stream = (df.withColumn("czas", col("timestamp"))
        .withColumn("temperatura", expr("20 + rand() * 10"))
        .select("czas", "temperatura")
       )

stream_anomaly = stream.withColumn(
    "anomaly",
    when(col("temperatura") > 29.5, "TAK").otherwise("NIE")
)

query = (stream_anomaly.writeStream 
    .format("console") 
    .outputMode("append")
    .foreachBatch(process_batch)
    .option("truncate", False) 
    .start()
)

In [ ]:
# 3. Okno czasowe: Tumbling Window

In [17]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import window

spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

def process_batch(df, batch_id, tstop=30):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()



df = (spark.readStream
      .format("rate")
      .option("rowsPerSecond", 1)
      .load()
)

stream = (df.withColumn("czas", col("timestamp"))
        .withColumn("temperatura", expr("20 + rand() * 10"))
        .select("czas", "temperatura")
       )
# Konwertuj 'temperatura' na typ Double, aby agregacja była możliwa
stream = stream.withColumn("temperatura", col("temperatura").cast("double"))

tumbling_window = (stream
    .groupBy(window(col("czas"), "10 seconds"))
    .avg("temperatura")
)

query = (tumbling_window.writeStream 
    .format("console") 
    .outputMode("complete")
    .foreachBatch(process_batch)
    .option("truncate", False) 
    .start()
)

In [ ]:
#  4. Okno czasowe: Sliding Window

In [18]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import window

spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

def process_batch(df, batch_id, tstop=30):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()



df = (spark.readStream
      .format("rate")
      .option("rowsPerSecond", 1)
      .load()
)

stream = (df.withColumn("czas", col("timestamp"))
        .withColumn("temperatura", expr("20 + rand() * 10"))
        .select("czas", "temperatura")
       )
# Konwertuj 'temperatura' na typ Double, aby agregacja była możliwa
stream = stream.withColumn("temperatura", col("temperatura").cast("double"))


sliding = (stream.groupBy(window(col("czas"), "10 seconds", "5 seconds"))
           .avg("temperatura")
          )

query = (sliding.writeStream 
    .format("console") 
    .outputMode("complete")
    .foreachBatch(process_batch)
    .option("truncate", False) 
    .start()
)

In [ ]:
# generator.py
import json, os, random, time
from datetime import datetime

output_dir = "data/stream"
os.makedirs(output_dir, exist_ok=True)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': round(random.uniform(5.0, 5000.0), 2),
        'store': random.choice(sklepy),
        'category': random.choice(kategorie),
        'timestamp': datetime.now().isoformat(),
    }

while True:
    batch = [generate_transaction() for _ in range(2)]
    filename = f"{output_dir}/events_{int(time.time())}.json"
    with open(filename, "w") as f:
        for e in batch:
            f.write(json.dumps(e) + "\n")
    print(f"Wrote: {filename}")
    time.sleep(5)


Wrote: data/stream/events_1776772081.json
Wrote: data/stream/events_1776772086.json
Wrote: data/stream/events_1776772091.json
Wrote: data/stream/events_1776772096.json
Wrote: data/stream/events_1776772101.json
Wrote: data/stream/events_1776772106.json
Wrote: data/stream/events_1776772111.json
Wrote: data/stream/events_1776772116.json
Wrote: data/stream/events_1776772121.json
Wrote: data/stream/events_1776772126.json
Wrote: data/stream/events_1776772131.json
Wrote: data/stream/events_1776772136.json
Wrote: data/stream/events_1776772141.json
Wrote: data/stream/events_1776772146.json
Wrote: data/stream/events_1776772151.json
Wrote: data/stream/events_1776772156.json
Wrote: data/stream/events_1776772161.json
Wrote: data/stream/events_1776772166.json
Wrote: data/stream/events_1776772171.json
Wrote: data/stream/events_1776772176.json
Wrote: data/stream/events_1776772182.json
Wrote: data/stream/events_1776772187.json
Wrote: data/stream/events_1776772192.json
Wrote: data/stream/events_17767721

In [ ]:
python generator.py


In [ ]:
stream = (spark.readStream
          .schema(tx_schema)
          .json("data/stream"))


In [ ]:
mkdir -p data/stream


In [ ]:
!mkdir -p data/stream


In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("jsonDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

tx_schema = StructType([
    StructField("tx_id",     StringType()),
    StructField("user_id",   StringType()),
    StructField("amount",    DoubleType()),
    StructField("store",     StringType()),
    StructField("category",  StringType()),
    StructField("timestamp", StringType()),
])


In [ ]:
stream = (spark.readStream
          .schema(tx_schema)
          .json("data/stream"))


In [ ]:
def process_batch(df, batch_id):
    print(f"\n=== Batch {batch_id} ===")
    df.show(truncate=False)


In [ ]:
query.awaitTermination()
